# This notebook is for abalation studies of HQNN without the quantum circuit to find the contribution of QC to the HQNN

# Setup

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Data Prep

## NIFTY

In [1]:
# NIFTY DATA
import pandas as pd
NIFTY = pd.read_csv(r'approach_3_C+P.csv')
NIFTY.sort_values(by='Date', inplace=True)

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Feature and target columns
feature_columns = [
    'Time to Maturity', 'Risk-Free Rate', 'Moneyness', 'mean_moneyness', 'previous_option_price_scaled',
    'mean_log_return_open', 'mean_log_return_high', 'mean_log_return_low', 'mean_log_return_close',
    'sqrt_var_open', 'sqrt_var_high', 'sqrt_var_low', 'sqrt_var_close', 'sqrt_cov_open_high',
    'sqrt_cov_open_low', 'sqrt_cov_open_close', 'sqrt_cov_high_low', 'sqrt_cov_high_close', 'sqrt_cov_low_close'
]
target_column = 'Bin'  # 'Bin_value' is not used here; add if needed

def prepare_data(approach_3_df, sample_size, batch_size=64):
    # Combine features and target
    data = approach_3_df[feature_columns + [target_column]]

    # Split into train, val, test
    train_df, temp_df = train_test_split(data, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    # Scale only the features
    scaler = StandardScaler()
    train_df.loc[:, feature_columns] = scaler.fit_transform(train_df[feature_columns])
    val_df.loc[:, feature_columns] = scaler.transform(val_df[feature_columns])
    test_df.loc[:, feature_columns] = scaler.transform(test_df[feature_columns])

    # Separate features and targets
    train_features = train_df[feature_columns]
    val_features = val_df[feature_columns]
    test_features = test_df[feature_columns]

    train_labels = torch.tensor(train_df[target_column].values, dtype=torch.float32, device=device).unsqueeze(1)
    val_labels = torch.tensor(val_df[target_column].values, dtype=torch.float32, device=device).unsqueeze(1)
    test_labels = torch.tensor(test_df[target_column].values, dtype=torch.float32, device=device).unsqueeze(1)

    # Convert features to tensors
    xtrain = torch.tensor(train_features.values, dtype=torch.float32, device=device)
    xval = torch.tensor(val_features.values, dtype=torch.float32, device=device)
    xtest = torch.tensor(test_features.values, dtype=torch.float32, device=device)

    # Create TensorDatasets
    train_dataset = TensorDataset(xtrain, train_labels)
    val_dataset = TensorDataset(xval, val_labels)
    test_dataset = TensorDataset(xtest, test_labels)

    # Create DataLoader instances
    gpu_loader_config = {
        "pin_memory": False,
        "num_workers": 0,
        "persistent_workers": False,
    }

    # Dynamically limit datasets to at most 1000 samples
    train_subset = Subset(train_dataset, range(min(sample_size, len(train_dataset))))
    val_subset = Subset(val_dataset, range(min(sample_size, len(val_dataset))))
    test_subset = Subset(test_dataset, range(min(sample_size, len(test_dataset))))

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, **gpu_loader_config)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, **gpu_loader_config)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, **gpu_loader_config)

    return train_loader, val_loader, test_loader


Using device: cuda


## SYNTHETIC

In [1]:
# SYNTHETIC DATA
import pandas as pd
SYN = pd.read_csv(r'synthetic_data.csv')

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Feature and target columns
feature_columns = [
    'feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9',
       'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14',
       'feature_15', 'feature_16', 'feature_17', 'feature_18']
target_column = 'target'  # 'Bin_value' is not used here; add if needed

def prepare_data(approach_3_df, sample_size, batch_size=64):
    # Combine features and target
    data = approach_3_df[feature_columns + [target_column]]

    # Split into train, val, test
    train_df, temp_df = train_test_split(data, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    # Scale only the features
    scaler = StandardScaler()
    train_df.loc[:, feature_columns] = scaler.fit_transform(train_df[feature_columns])
    val_df.loc[:, feature_columns] = scaler.transform(val_df[feature_columns])
    test_df.loc[:, feature_columns] = scaler.transform(test_df[feature_columns])

    # Separate features and targets
    train_features = train_df[feature_columns]
    val_features = val_df[feature_columns]
    test_features = test_df[feature_columns]

    train_labels = torch.tensor(train_df[target_column].values, dtype=torch.float32, device=device).unsqueeze(1)
    val_labels = torch.tensor(val_df[target_column].values, dtype=torch.float32, device=device).unsqueeze(1)
    test_labels = torch.tensor(test_df[target_column].values, dtype=torch.float32, device=device).unsqueeze(1)

    # Convert features to tensors
    xtrain = torch.tensor(train_features.values, dtype=torch.float32, device=device)
    xval = torch.tensor(val_features.values, dtype=torch.float32, device=device)
    xtest = torch.tensor(test_features.values, dtype=torch.float32, device=device)

    # Create TensorDatasets
    train_dataset = TensorDataset(xtrain, train_labels)
    val_dataset = TensorDataset(xval, val_labels)
    test_dataset = TensorDataset(xtest, test_labels)

    # Create DataLoader instances
    gpu_loader_config = {
        "pin_memory": False,
        "num_workers": 0,
        "persistent_workers": False,
    }

    # Dynamically limit datasets to at most 1000 samples
    train_subset = Subset(train_dataset, range(min(sample_size, len(train_dataset))))
    val_subset = Subset(val_dataset, range(min(sample_size, len(val_dataset))))
    test_subset = Subset(test_dataset, range(min(sample_size, len(test_dataset))))

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, **gpu_loader_config)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, **gpu_loader_config)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, **gpu_loader_config)

    return train_loader, val_loader, test_loader


Using device: cuda


# Model definition

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import pennylane as qml
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import time
import random
import os
import json
import csv

In [ ]:
def run():
    # =====================
    # 2. GPU Configuration
    # =====================
    if USE_GPU and not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. This script requires a GPU.")

    device = torch.device("cuda" if USE_GPU else "cpu")
    print(f"Using device: {device}")
    print(f"PyTorch CUDA version: {torch.version.cuda}")

    # Quantum device setup with fixed settings
    try:
        dev = qml.device("lightning.gpu", wires=NUM_QUBITS, batch_obs=True, shots=None)
        print(f"PennyLane device configured: {dev.name} on GPU")
    except Exception as e:
        print(f"Error initializing PennyLane lightning.gpu device: {e}")
        exit()

    # =====================
    # 3. Quantum Circuit
    # =====================

    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def quantum_circuit(inputs, weights):
        for i in range(NUM_QUBITS):
            qml.RY(inputs[:, i], wires=i)

        for layer in range(NUM_LAYERS):
            for i in range(NUM_QUBITS):
                qml.RZ(weights[layer, i], wires=i)
            for i in range(NUM_QUBITS):
                qml.CNOT(wires=[i, (i + 1) % NUM_QUBITS])

        return [qml.expval(qml.PauliZ(i)) for i in range(NUM_QUBITS)]

    # =====================
    # 4. Classical Components
    # =====================
    class PreAutoencoder(nn.Module):
        def __init__(self, input_dim=INPUT_DIM, output_dim=NUM_QUBITS):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Linear(HIDDEN_DIM, output_dim),
                nn.Tanh()
            )

        def forward(self, x):
            return self.encoder(x)

    class PostAutoencoder(nn.Module):
        def __init__(self, input_dim=NUM_QUBITS, output_dim=FINAL_OUTPUT_DIM):
            super().__init__()
            self.decoder = nn.Sequential(
                nn.Linear(input_dim, POST_HIDDEN_DIM),
                nn.ReLU(),
                nn.Linear(POST_HIDDEN_DIM, output_dim)
            )

        def forward(self, x):
            return self.decoder(x)

    # Quantum layer
    weight_shapes = {"weights": (NUM_LAYERS, NUM_QUBITS)}
    qnn_model = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

    # =====================
    # 5. Hybrid Model
    # =====================
    class HybridQNN(nn.Module):
        def __init__(self, pre_encoder, qnn, post_decoder):
            super().__init__()
            self.pre_encoder = pre_encoder
            #self.quantum_layer = qnn
            self.post_decoder = post_decoder
            self.to(device)

        def forward(self, x):
            x = x.to(device)

            with torch.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                encoded = self.pre_encoder(x)
                scaled_inputs = encoded * math.pi
                #quantum_out = self.quantum_layer(scaled_inputs).float()
                #quantum_out = quantum_out.to(device)
                #return self.post_decoder(quantum_out)
                return self.post_decoder(scaled_inputs)
            
    # ========== Initialize Model ==========
    pre_ae = PreAutoencoder()
    post_ae = PostAutoencoder()
    model = HybridQNN(pre_ae, qnn_model, post_ae)

    # ===========================
    # 6. Training Loop Setup
    # ===========================
    train_loader, val_loader, test_loader = prepare_data(approach_3_df=DATA, sample_size=SAMPLE_SIZE, batch_size=BATCH_SIZE)

    # =====================
    # 7. Optimizer Setup
    # =====================
    optimizer = optim.AdamW([
        {'params': model.pre_encoder.parameters(), 'lr': LR_PRE_ENCODER, 'weight_decay': WEIGHT_DECAY},
        #{'params': model.quantum_layer.parameters(), 'lr': LR_QUANTUM},
        {'params': model.post_decoder.parameters(), 'lr': LR_POST_DECODER, 'weight_decay': WEIGHT_DECAY}
    ])

    # =====================
    # 8. Learning Rate Scheduler
    # =====================
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=LR_SCHEDULER_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
        verbose=True
    )

    # =====================
    # 9. Mixed Precision Support
    # =====================
    scaler = torch.GradScaler(device=device)

    # =====================
    # 10. Loss Function
    # =====================
    criterion = nn.MSELoss()

    # =====================
    # 11. Training Loop 
    # =====================
    best_val_loss = float('inf')
    epochs_no_improve = 0
    train_losses = []
    val_losses = []
    gradient_norms_total = []
    gradient_norms_per_layer = []
    learning_rates = []
    best_epoch = 0

    start_time = time.time()

    training_logs = []

    # Start Training Loop
    for epoch in tqdm(range(NUM_EPOCHS), desc="Training Progress", unit="epoch"):

        # === Training ===
        model.train()
        train_loss = 0.0
        epoch_grad_per_layer = {'pre_encoder': 0.0, 'quantum_layer': 0.0, 'post_decoder': 0.0}

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            with torch.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)  # Unscale before computing gradient norm

            # === Compute Gradient Norms ===
            total_norm = 0.0
            for name, module in model.named_children():
                layer_norm = 0.0
                for param in module.parameters():
                    if param.grad is not None:
                        param_norm = param.grad.data.norm(2)
                        total_norm += param_norm.item() ** 2
                        layer_norm += param_norm.item() ** 2
                epoch_grad_per_layer[name] += layer_norm ** 0.5

            total_norm = total_norm ** 0.5
            gradient_norms_total.append(total_norm)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item() * inputs.size(0)

        # === Average Gradient Norm Per Layer (per epoch) ===
        gradient_norms_per_layer.append({
            layer: norm for layer, norm in epoch_grad_per_layer.items()
        })

        # === Validation ===
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

        # === Update Epoch Metrics ===
        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        learning_rates.append(optimizer.param_groups[0]['lr'])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), SAVE_MODEL_PATH)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            break

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Save logs for this epoch
        training_logs.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "gradient_norm_total": total_norm,
            "gradient_norm_layerwise": epoch_grad_per_layer,
            "learning_rate": optimizer.param_groups[0]["lr"]
        })

    training_time = (time.time() - start_time) / 60  # in minutes

    # ===========================
    # Final Metrics Logging
    # ===========================
    final_train_loss = train_losses[-1]
    final_val_loss = val_losses[-1]
    generalization_gap = final_val_loss - final_train_loss
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ===========================
    # Save Final Predictions
    # ===========================
    model.eval()
    final_predictions = []
    final_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            final_predictions.append(outputs.cpu().numpy())
            final_labels.append(labels.cpu().numpy())

    # Convert to numpy arrays for easy saving
    final_predictions = np.concatenate(final_predictions, axis=0)
    final_labels = np.concatenate(final_labels, axis=0)

    # Save the predictions and labels to CSV
    os.makedirs('logs', exist_ok=True)
    np.savetxt(f'{DATA_NAME}/{DATA_NAME}_{NUM_SAMPLES}/logs/{model_name}_predictions.csv', final_predictions, delimiter=',')
    np.savetxt(f'{DATA_NAME}/{DATA_NAME}_{NUM_SAMPLES}/logs/{model_name}_labels.csv', final_labels, delimiter=',')

    # ===========================
    # Save Training Logs to JSON and CSV
    # ===========================
    os.makedirs("logs", exist_ok=True)

    # Save training logs to JSON
    with open(f"{DATA_NAME}/{DATA_NAME}_{NUM_SAMPLES}/logs/{model_name}_logs.json", "w") as f:
        json.dump(training_logs, f, indent=4)

    # Save summary to CSV
    run_summary = {
        "model_id": model_name,
        "architecture": " ",
        "qubits": NUM_QUBITS,
        "depth": NUM_LAYERS,
        "batch_size": BATCH_SIZE,
        "sample_size": NUM_SAMPLES,
        "random_seed": RANDOM_SEED,
        "train_loss_final": final_train_loss,
        "val_loss_final": final_val_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "final_grad_norm": training_logs[-1]["gradient_norm_total"],
        "generalization_gap": generalization_gap,
        "training_time_minutes": training_time,
        "num_params": num_params
    }

    csv_path = f"{DATA_NAME}/{DATA_NAME}_{NUM_SAMPLES}/logs/{model_name}_run_summaries.csv"
    write_headers = not os.path.exists(csv_path)
    with open(csv_path, "a", newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=run_summary.keys())
        if write_headers:
            writer.writeheader()
        writer.writerow(run_summary)



## Training Real world data

In [ ]:
import torch.backends.cudnn as cudnn

l = [42,52,62]  # random seeds

data_sample_size = [1000] 

config = [(2,2),(2,3),(2,4),(2,5),
          (3,2),(3,3),(3,4),(3,5),
          (4,2),(4,3),(4,4),(4,5),
          (5,2),(5,3),(5,4),(5,5)]

c = 0

datasets = [("ANN_NIFTY", NIFTY)]

for sample_size in data_sample_size:
    for num_qubit, layers in config:
        c += 1

        for DATA_NAME, DATA in datasets: 


            for rs in l:
                # General settings
                SAMPLE_SIZE = sample_size
                USE_GPU = True
                BATCH_SIZE = 32
                INPUT_DIM = 19
                HIDDEN_DIM = 8
                POST_HIDDEN_DIM = 40
                FINAL_OUTPUT_DIM = 1
                NUM_SAMPLES = SAMPLE_SIZE
                # Quantum settings
                NUM_QUBITS = num_qubit
                NUM_LAYERS = layers
                RANDOM_SEED = rs
                DIFF_METHOD = "adjoint"

                # Reproducibility
                torch.manual_seed(RANDOM_SEED)
                torch.cuda.manual_seed_all(RANDOM_SEED)
                np.random.seed(RANDOM_SEED)
                random.seed(RANDOM_SEED)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(RANDOM_SEED)
                cudnn.deterministic = True
                cudnn.benchmark = False
                # Model name
                model_name = f"{DATA_NAME}_{sample_size}_{c}_{rs}"

                # Print configuration
                print(f"Running {model_name} | Dataset: {DATA_NAME} | "
                      f"Sample Size: {SAMPLE_SIZE} | Qubits: {NUM_QUBITS} | "
                      f"Layers: {NUM_LAYERS} | Seed: {RANDOM_SEED}")

                # Training settings
                NUM_EPOCHS = 100
                EARLY_STOP_PATIENCE = 5
                LR_PRE_ENCODER = 0.001
                LR_QUANTUM = 0.0001
                LR_POST_DECODER = 0.001
                WEIGHT_DECAY = 0.01
                LR_SCHEDULER_FACTOR = 0.5
                LR_SCHEDULER_PATIENCE = 3
                SAVE_MODEL_PATH = f"{model_name}.pth"
                # Run training
                run()
                    
print(f"\nTotal runs: {c}")


## Training synthetic data

In [ ]:
# Classical models for nifty data
import torch.backends.cudnn as cudnn

l = [42,52,62]  # random seeds

data_sample_size = [1000] 

config = [(2,2),(2,3),(2,4),(2,5),
          (3,2),(3,3),(3,4),(3,5),
          (4,2),(4,3),(4,4),(4,5),
          (5,2),(5,3),(5,4),(5,5)]


c = 0

# Dataset loop with (name, dataframe)
datasets = [("ANN_SYN", SYN)]

for sample_size in data_sample_size:
    for num_qubit, layers in config:
        c += 1

        for DATA_NAME, DATA in datasets:  # SNP500 first, then NIFTY


            for rs in l:
                # General settings
                SAMPLE_SIZE = sample_size
                USE_GPU = True
                BATCH_SIZE = 32
                INPUT_DIM = 19
                HIDDEN_DIM = 8
                POST_HIDDEN_DIM = 40
                FINAL_OUTPUT_DIM = 1
                NUM_SAMPLES = SAMPLE_SIZE
                # Quantum settings
                NUM_QUBITS = num_qubit
                NUM_LAYERS = layers
                RANDOM_SEED = rs
                DIFF_METHOD = "adjoint"

                # Reproducibility
                torch.manual_seed(RANDOM_SEED)
                torch.cuda.manual_seed_all(RANDOM_SEED)
                np.random.seed(RANDOM_SEED)
                random.seed(RANDOM_SEED)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(RANDOM_SEED)
                cudnn.deterministic = True
                cudnn.benchmark = False
                # Model name
                model_name = f"{DATA_NAME}_{sample_size}_{c}_{rs}"

                # Print configuration
                print(f"Running {model_name} | Dataset: {DATA_NAME} | "
                      f"Sample Size: {SAMPLE_SIZE} | Qubits: {NUM_QUBITS} | "
                      f"Layers: {NUM_LAYERS} | Seed: {RANDOM_SEED}")

                # Training settings
                NUM_EPOCHS = 100
                EARLY_STOP_PATIENCE = 5
                LR_PRE_ENCODER = 0.001
                LR_QUANTUM = 0.0001
                LR_POST_DECODER = 0.001
                WEIGHT_DECAY = 0.01
                LR_SCHEDULER_FACTOR = 0.5
                LR_SCHEDULER_PATIENCE = 3
                SAVE_MODEL_PATH = f"{model_name}.pth"
                # Run training
                run()
                    
print(f"\nTotal runs: {c}")
